# Create GRA Awards (Georgia Research Alliance)

Creates Georgia Research Alliance awards from GRA's own research-talent
listings (scraped by `scripts/local/gra_to_s3.py`; gra.org robots.txt blocks
nothing). GRA publishes no grants database; per person it publishes:

- **GRA Eminent Scholars** (63): endowed chairs GRA funds at Georgia's research
  universities (research area, university, year recruited, bio)
- **GRA Distinguished Investigators** (9): research area, university, year recruited
- **GRA Senior Fellows** (6): role, centre, university (no year)

**78 awards, recruited 1993-2026.** One row per person. `display_name` is
composed (`GRA Eminent Scholar: <name> (<research area>, <university>)`),
`description` = GRA's bio, lead investigator = the scholar with the university
as affiliation, `start_year` = year recruited. Only current listings are
online: 33 former scholars' pages now redirect away, so former Eminent
Scholars are not covered.

**Not inserted (flagged, kept in the raw table):** the 212 GRA portfolio
companies (`record_type = 'portfolio_company'`, gra.org/company/{id}/) --
startups GRA's Innovation & Entrepreneurship programme backs with loans /
venture funding. They carry no grant, amount, date or investigator, only a
company name and blurb, so they are not inserted as awards. To include them,
drop the `record_type <> 'portfolio_company'` filter in Step 2 (adding rows
later is safe; removing published rows is not). GRA's I&E university grants
(cited as `GRA.VL19.C3` etc.) are not published by GRA, so they cannot be
ingested.

**`funder_award_id`:** GRA publishes no numbers for these, so stable synthetic
keys from the page ids: `GRA-ES-{id}`, `GRA-DI-{id}`, `GRA-SF-{id}`. No
existing citation stub uses these forms (0 collapse).

**Amount:** not published (§6.7 waived: endowed chairs/recruitment awards
with no per-award amounts).

**Prerequisites:** run `scripts/local/gra_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/gra/gra_projects.parquet` (§1.4 shrink guard).

**Funder details (F4320* path):**
- funder_id: `4320310224`
- display_name: Georgia Research Alliance
- ror_id / doi: from `openalex.funders.funders` (ror 030689123, doi 10.13039/100008065)

**Priority:** `595` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gra_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/gra/gra_projects.parquet`;

In [ ]:
%sql
SELECT record_type, COUNT(*) as n
FROM openalex.awards.gra_raw
GROUP BY 1 ORDER BY 1;

In [ ]:
%sql
DESCRIBE openalex.awards.gra_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.gra_raw LIMIT 5;

## Step 1.6: Funder existence check (F4320* path)
F4320310224 must resolve to exactly 1 row in `openalex.funders.funders`.
If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320310224;

## Step 2: Create GRA Awards Table
Portfolio companies are excluded here (see header).

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gra_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320310224  -- Georgia Research Alliance
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    g.funder_scheme as funder_scheme,
    'gra_research_talent' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.recruited_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    CASE WHEN NULLIF(TRIM(g.lead_family_name), '') IS NOT NULL THEN named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', TRIM(g.lead_family_name),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.organization), ''),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>)
        as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>)
        as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.gra_raw g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != ''
  AND g.record_type <> 'portfolio_company';

In [ ]:
%sql
-- Shape check: synthetic keys only. Expect 0 rows.
SELECT funder_award_id, display_name
FROM openalex.awards.gra_awards
WHERE NOT funder_award_id RLIKE '^GRA-(ES|DI|SF)-[0-9]+$';

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'gra_research_talent' AND priority = 595;

-- Insert into openalex_awards_raw with priority.
-- Priority 595: direct-from-funder ingest of GRA's research-talent listings.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    595 as priority
FROM openalex.awards.gra_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_gra_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.gra_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, landing_page_url
FROM openalex.awards.gra_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, MIN(start_year) as first_year, MAX(start_year) as last_year
FROM openalex.awards.gra_awards
GROUP BY 1 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name as university, COUNT(*) as n
FROM openalex.awards.gra_awards
GROUP BY 1 ORDER BY n DESC;

In [ ]:
%sql
-- Section 6.4a frequency check: one row per person, so every count should be 1.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.gra_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.gra_awards
GROUP BY 1 ORDER BY n DESC LIMIT 10;

In [ ]:
%sql
-- Section 6.3 coverage (expect 100% title/description/PI/university, 92% start_year, 0% amount).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.gra_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'gra_research_talent'
GROUP BY provenance, priority;